# 학습 데이터셋 결정: 705명 vs 원래 445명 출신

| 순서 | 노트북 | 내용 |
|---|---|---|
| 0 | `ml_0_dataset` | 학습 데이터셋 결정 (705명 vs 원래 445명 출신 → 445명). class weight를 Optuna로 함께 튜닝 (결과 `kfold_logloss`) |
| 1 | `ml_1_main` | class weight 고정으로 학습하고 모델 비교 (결과 `kfold_logloss_balanced`). 6번에서 최종 모델 1개 선택 |
| 2 | `ml_2_test_all_models` | 참고: 0·1의 445명 모델 전체 test 성능 (모델 선택에 쓰지 않음) |
| 3 | `ml_3_final_test` | 최종 모델 1개 재학습과 test 1회 평가 (최종 모델을 고른 뒤 실행) |

**가장 먼저 실행합니다.** 개발 705명 중 349명은 원래 445명 코호트 밖에서 추가한 환자입니다(결측이 이 환자들에게만 있고, 기기 이식 비율 28%). 이 환자들을 학습에 넣을지 정합니다.

`ml_1_main`과 같은 구성(기준 모델 + 모델 4종 × 변수 묶음 4종)을 학습 환자 2종으로 각각 학습합니다.

| 학습 환자 | fold당 학습 | 내용 |
|---|---|---|
| `all` | 564명 | 705명 전체 |
| `only445` | 약 285명 | 원래 445명 출신만 |

- `only445`는 **학습, 튜닝(log loss), 평가를 모두 원래 445명 출신 356명 안에서** 합니다. fold는 같고, 그중 356명만 씁니다.
- 비교는 두 방법 모두 예측이 있는 356명(SCD 40명)에서 합니다. test 89명과 같은 집단입니다. `all`은 705명으로 튜닝한 그대로입니다.
- 이미 학습된 방법은 건너뜁니다. 여기서 학습한 445명 모델 16개는 `ml_2_test_all_models`에서 test 성능을 기록합니다.


## 0. 준비

In [ ]:
import contextlib
import io
import sys
from pathlib import Path

import pandas as pd

# 저장소 루트를 찾아 import 경로에 추가합니다 (노트북을 어느 폴더에서 열어도 동작).
repo = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Members/mingyu/model/ML/kfold").is_dir()
)
sys.path.insert(0, str(repo))

from Members.mingyu.model.ML.kfold import (  # noqa: E402
    Method,
    Settings,
    compare,
    describe,
    load_cohort,
    run_method,
    save_or_check_folds,
)  # noqa: E402

# 데이터셋 결정은 class weight를 Optuna로 튜닝하던 이전 실행(kfold_logloss)에서 했습니다.
settings = Settings(run_tag="kfold_logloss", class_weight=None)
cohort = load_cohort(settings)
save_or_check_folds(cohort, settings)  # 처음엔 fold를 저장하고, 이후엔 같은 fold인지 확인

print(
    f"개발 {len(cohort.ids)}명 (SCD {cohort.y.sum()}) | 그중 원래 445명 출신 {cohort.original.sum()}명 | test {len(cohort.test_ids)}명"
)
print("결과 폴더:", settings.result_dir)

## 1. 판정 규칙 (결과를 보기 전에 정함)

- **기본은 445명입니다.** test와 같은 집단이고, 결측이 없고, SCD 라벨의 의미가 같습니다.
- 16개 조합마다 445명 출신 356명에서 ΔAUROC(`445명 학습 − 705명 학습`)의 95% CI를 봅니다. CI 상한이 0보다 작으면 "705명이 유의하게 나음"입니다.
- **"705명이 유의하게 나음"이 16개 중 8개 이상이면 705명**, 아니면 445명을 씁니다.


## 2. 학습 설정

| 변수 묶음 | 개수 | 내용 |
|---|---|---|
| `v5` | 27 | 팀 임상 변수 23 + Holter 4 |
| `v4` | 23 | 팀 기본 임상 변수 |
| `core` | 12 | 주요 임상 변수 + Holter 3 |
| `MUSIC_SCD` | 5 | MUSIC SCD 위험 점수 항목 |

- 튜닝: 로지스틱은 C 9개, 트리 모델은 Optuna 100 trial. 기준은 OOF log loss
- class weight: 고정하지 않음. 트리 모델은 Optuna가 class weight도 함께 탐색, 로지스틱은 class weight 없음 (결과 폴더 `kfold_logloss`)
- 학습 환자마다 따로 튜닝합니다. 튜닝 기준 log loss도 그 데이터셋의 환자로 계산합니다 (`all` = 705명, `only445` = 356명).


In [ ]:
FEATURE_SETS_DATASET = ["v5", "v4", "core", "MUSIC_SCD"]
MODELS_DATASET = ["Logistic", "XGBoost", "Random Forest", "CatBoost"]

## 3. 705명 전체로 학습 (`all`)

fold당 학습 환자 564명

### 3-0. 기준 모델: Pro-BNP 단독 로지스틱


In [ ]:
reference_705 = run_method(cohort, Method("Logistic", "ProBNP", "all"), settings)
describe(settings, cohort, [reference_705])

### 3-1. Logistic

C 후보 9개를 모두 시험합니다.


In [ ]:
logistic_705 = [
    run_method(cohort, Method("Logistic", features, "all"), settings)
    for features in FEATURE_SETS_DATASET
]
describe(settings, cohort, logistic_705)

### 3-2. XGBoost

Optuna 100 trial × 15 fold.


In [ ]:
xgboost_705 = [
    run_method(cohort, Method("XGBoost", features, "all"), settings)
    for features in FEATURE_SETS_DATASET
]
describe(settings, cohort, xgboost_705)

### 3-3. Random Forest

Optuna 100 trial × 15 fold.


In [ ]:
random_forest_705 = [
    run_method(cohort, Method("Random Forest", features, "all"), settings)
    for features in FEATURE_SETS_DATASET
]
describe(settings, cohort, random_forest_705)

### 3-4. CatBoost

Optuna 100 trial × 15 fold.


In [ ]:
catboost_705 = [
    run_method(cohort, Method("CatBoost", features, "all"), settings)
    for features in FEATURE_SETS_DATASET
]
describe(settings, cohort, catboost_705)

## 4. 원래 445명 출신만 학습 (`only445`)

fold당 학습 환자 약 285명. 튜닝과 평가도 445명 출신 356명 안에서 합니다.

### 4-0. 기준 모델: Pro-BNP 단독 로지스틱


In [ ]:
reference_445 = run_method(cohort, Method("Logistic", "ProBNP", "only445"), settings)
describe(settings, cohort, [reference_445])

### 4-1. Logistic

C 후보 9개를 모두 시험합니다.


In [ ]:
logistic_445 = [
    run_method(cohort, Method("Logistic", features, "only445"), settings)
    for features in FEATURE_SETS_DATASET
]
describe(settings, cohort, logistic_445)

### 4-2. XGBoost

Optuna 100 trial × 15 fold.


In [ ]:
xgboost_445 = [
    run_method(cohort, Method("XGBoost", features, "only445"), settings)
    for features in FEATURE_SETS_DATASET
]
describe(settings, cohort, xgboost_445)

### 4-3. Random Forest

Optuna 100 trial × 15 fold.


In [ ]:
random_forest_445 = [
    run_method(cohort, Method("Random Forest", features, "only445"), settings)
    for features in FEATURE_SETS_DATASET
]
describe(settings, cohort, random_forest_445)

### 4-4. CatBoost

Optuna 100 trial × 15 fold.


In [ ]:
catboost_445 = [
    run_method(cohort, Method("CatBoost", features, "only445"), settings)
    for features in FEATURE_SETS_DATASET
]
describe(settings, cohort, catboost_445)

## 5. 판정표: 445명 출신 356명

조합마다 705명 학습을 기준(Δ=0)으로 놓은 paired bootstrap 결과입니다. `comparison_dataset_original.csv`로 저장됩니다.

`only445`는 추가 환자 349명에게 예측이 없으므로 705명 평가 표는 만들지 않습니다.


In [ ]:
def dataset_table(population: str) -> pd.DataFrame:
    """One row per model x feature set: 705 vs 445 training, evaluated on the same 356 patients."""
    rows = []
    for model in MODELS_DATASET:
        for features in FEATURE_SETS_DATASET:
            full, small = Method(model, features).name, Method(model, features, "only445").name
            with contextlib.redirect_stdout(
                io.StringIO()
            ):  # compare prints one header line per call
                table = compare(
                    settings, cohort, [full, small], reference=full, population=population
                ).set_index("방법")
            delta = table.loc[small, "ΔAUROC (95% CI)"]
            low, high = (float(v) for v in delta.split("(")[1].rstrip(")").split("~"))
            rows.append(
                {
                    "모델": model,
                    "변수": features,
                    "AUROC 705명 학습": table.loc[full, "AUROC"],
                    "AUROC 445명 학습": table.loc[small, "AUROC"],
                    "ΔAUROC 445−705 (95% CI)": delta,
                    "판정": "705명 유의하게 나음"
                    if high < 0
                    else "445명 유의하게 나음"
                    if low > 0
                    else "차이 없음",
                }
            )
    result = pd.DataFrame(rows)
    result.to_csv(settings.result_dir / f"comparison_dataset_{population}.csv", index=False)
    return result


judged = dataset_table("original")
display(judged)
n_705 = (judged["판정"] == "705명 유의하게 나음").sum()
print(
    f"705명이 유의하게 나은 조합: {n_705} / {len(judged)} → 학습 데이터셋: {'705명 (all)' if n_705 >= 8 else '445명 출신 (only445)'}"
)

## 6. 결정

- 학습 데이터셋: **원래 445명 출신 (`only445`)**
- 근거: 445명 출신 356명에서 705명 학습이 유의하게 나은 조합 0 / 16 (사전 규칙: 8개 이상이면 705명). 모두 95% CI가 0을 포함. 방향은 705명 쪽 8개, 445명 쪽 8개로 반반. XGBoost MUSIC_SCD가 가장 705명 쪽 (−0.022, CI −0.045 ~ +0.001) (2026-11-03, only445를 356명 log loss로 튜닝한 결과)

정한 값을 `ml_1_main`의 `TRAIN_FILTER`에 적습니다 (`"all"` = 705명, `"only445"` = 445명 출신). 445명이면 최종 모델 비교도 445명 출신 356명 기준으로 합니다.
